In [3]:
import pandas as pd

train_base = pd.read_csv("/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/train/train_base.csv", low_memory=False)
test_base = pd.read_csv("/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/test/test_base.csv", low_memory=False)


# Load both pieces or files of the static_0 training table
static_train_0 = pd.read_csv(
    "/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/train/train_static_0_0.csv",
    low_memory=False
)

static_train_1 = pd.read_csv(
    "/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/train/train_static_0_1.csv",
    low_memory=False
)

# Stack or combine both pieces or files into one complete training static table
static_train = pd.concat(
    [static_train_0, static_train_1],
    ignore_index=True
)

# Load both pieces of the static_0 test table
static_test_0 = pd.read_csv(
    "/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/test/test_static_0_0.csv",
    low_memory=False
)

static_test_1 = pd.read_csv(
    "/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/test/test_static_0_1.csv",
    low_memory=False
)

# Stack both pieces into one complete test static table
static_test = pd.concat(
    [static_test_0, static_test_1],
    ignore_index=True
)
train = train_base.merge(
    static_train,
    on="case_id",
    how="left",
    validate="one_to_one"
)

test = test_base.merge(
    static_test,
    on="case_id",
    how="left",
    validate="one_to_one"
)


print(train.shape)
print(test.shape)

(1000000, 12)
(200000, 11)


In [4]:
features = [
    "mainoccupationinc_384A",
    "credamount_770A",
    "annuity_780A",
    "days_employed_700P",
    "education_927M",
    "maritalstatus_703M",
]

print("Missingness:")
print((train[features].isna().mean() * 100).round(1))

print("\nNumeric summary:")
print(
    train[
        [
            "mainoccupationinc_384A",
            "credamount_770A",
            "annuity_780A",
            "days_employed_700P",
        ]
    ].describe(percentiles=[0.01, 0.50, 0.99])
)

Missingness:
mainoccupationinc_384A     5.0
credamount_770A            2.0
annuity_780A              10.0
days_employed_700P        15.1
education_927M             0.0
maritalstatus_703M         0.0
dtype: float64

Numeric summary:
       mainoccupationinc_384A  credamount_770A   annuity_780A  \
count           950349.000000    980041.000000  900323.000000   
mean             24477.415393     16730.809971    1242.750132   
std              11911.971994      8932.874580     661.880406   
min               2515.000000      1447.000000      79.000000   
1%                7516.000000      4610.000000     342.000000   
50%              22021.000000     14760.000000    1097.000000   
99%              64417.560000     47330.000000    3505.000000   
max             174473.000000    150395.000000   12069.000000   

       days_employed_700P  
count       849429.000000  
mean          1524.346248  
std            902.202133  
min              0.000000  
1%               0.000000  
50%           

## EDA findings and cleaning rules

### Missing values would include the following:

- `mainoccupationinc_384A`: 5.0% missing
- `credamount_770A`: 2.0% missing
- `annuity_780A`: 10.0% missing
- `days_employed_700P`: 15.1% missing
- `education_927M` and `maritalstatus_703M`: 0% missing

### Numeric values and outliers

The numeric amount fields are right-skewed: their maximum values are much larger than their 99th-percentile values. For example, income has a 99th percentile of about 64,418 but a maximum of 174,473.

No rows will be removed or eliminated for this initial baseline. Numeric missing values will be median-imputed and numeric variables will use `RobustScaler`, which is less sensitive or responsive to extreme values.

In [ ]:
%pip install -q scikit-learn

from sklearn.model_selection import train_test_split

X = train[features].copy()
y = train["target"].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y) 
# For Stratify = y -> It keeps the same target ratio in train and validation.
# Example: if 20% of all loans default, both sets stay close to 20% defaults.

ModuleNotFoundError: No module named 'sklearn'